# needle-fp4: which matmul breaks 4-bit training, and what 4-bit weights save

## 1. Title and members

**CMU 10-414/714 Deep Learning Systems, final project. Group 59 (slot B6).**
Shreya Verma (shreyave), Anshuman Mishra (anshm), Nidhi Prakash (nidhipra).

This notebook is both our report and our code. **To reproduce:** Runtime > Run all. It works with or without a GPU
(Runtime > Change runtime type > GPU adds the CUDA kernel tests and the GPU benchmark). It builds needle, runs the tests,
re-runs the key training result, benchmarks the 4-bit GEMV, compares NVFP4 with MXFP4, and zips the output as
`fp4_results.zip`. Each experiment cell ends with a one-line **Verdict**. If the repository is private, add a
`GITHUB_TOKEN` secret first (see the setup cell).

*AI-assisted: written with Claude (Anthropic) via Claude Code; reviewed and tested by the team (see section 8).*

## 2. Introduction

New accelerators multiply 4-bit floating point numbers (NVIDIA's NVFP4). A 4-bit weight takes one eighth of the
memory of FP32, and text generation at batch size 1 is limited by how fast weights stream from memory, so smaller
weights mean faster tokens. Training in 4 bits is harder: an E2M1 number has only 15 distinct values, so every
operand of every matmul is rounded hard.

A linear layer runs three matmuls in training: the forward pass **Fprop** `Y = XW`, the input gradient
**Dgrad** `dX = dY Wᵀ`, and the weight gradient **Wgrad** `dW = Xᵀ dY`. Our hypothesis: the gradient matmuls are the
fragile ones, because round-to-nearest on small gradients is biased, and **stochastic rounding (SR)**, which rounds up
with probability equal to the fractional distance, removes that bias.

We built NVFP4 into needle, the framework from our homeworks (no PyTorch, TensorFlow, JAX or cuBLAS), from a NumPy
reference down to CUDA kernels, and found:

* **Dgrad is the matmul that breaks.** On a CIFAR-10 MLP, Dgrad with round-to-nearest costs
  7.5 accuracy points; with Fprop and Wgrad also in 4 bits, accuracy falls from
  44.0% to 23.3% and training loss rises. With SR every configuration is within
  0.4 points of FP32. Fprop, and Wgrad on its own,
  are harmless. Quantizing W consistently in the forward and backward pass, as recent NVFP4 recipes do, does not
  fix it; rounding the gradient stochastically does.
* **Real 4-bit weights are 7x smaller** (97.42% vs 97.54% MNIST accuracy), and our CUDA kernel multiplies
  directly from them.
* **Stretch goal:** NVFP4 stores weights with about 0.8 to 0.9 dB less error than MXFP4, for 0.25 more bits per weight.

## 3. Background and related work

**The formats.** E2M1 has a sign, 2 exponent bits and 1 mantissa bit: 0, 0.5, 1, 1.5, 2, 3, 4, 6 and their
negatives. Because that range is tiny, each small block of numbers along the matmul's reduction axis shares a scale.

| Format | Block | Block scale | Extra scale | Bits per weight |
|---|---|---|---|---|
| NVFP4 | 16 | E4M3 (8-bit float) | FP32 per tensor | 4.5 |
| MXFP4 (OCP MX) | 32 | E8M0 (power of two) | none | 4.25 |

For NVFP4 the tensor scale is amax / (6 · 448) and each block scale is (block amax / 6) / tensor scale rounded to
E4M3. Every operand is quantized along its own reduction axis, so each of the three matmuls quantizes different axes.

**4-bit training.** NVIDIA pretrained a 12B-parameter model on 10T tokens in NVFP4, keeping a few layers in higher
precision and rounding gradients stochastically [1]. Chmiel et al. [2] train fully in FP4 and argue that gradients
need SR, plus higher precision once gradient noise dominates. Tseng et al. [3] and Quartet [4] train in MXFP4 with SR.
DeepSeek-V3 [5] showed fine-grained block scaling (FP8) at scale.

**Where we fit.** Those works train large models with all the tricks at once. We isolate the three matmuls of a
linear layer one at a time, with and without SR, in a framework we built ourselves, and measure the gradient error
each one causes. The contribution is the system and that ablation, at small scale.

## 4. Software architecture and design

One rule drives the design: **a NumPy reference defines NVFP4 exactly, and every other layer must match it bit for
bit.**

```
           NumPy reference (fp4/reference.py): quantize, pack, dequantize, GEMV order
              |  bit-exact tests          |  bit-exact tests              |  bit-exact tests
   NDArray fake quant  --->  Autograd Op + QuantConfig        Packed NVFP4 + kernels (CPU, CUDA)
   (fp4/quantize.py)         (ops/ops_fp4.py, fp4/config.py)   (fp4/packed.py, src/nvfp4.h)
                                      |                                    |
                          nn.QuantizedLinear (training)          4-bit inference (GEMV)
```

| Piece | Files | What it does |
|---|---|---|
| Reference | `python/needle/fp4/reference.py` | NumPy quantize, pack, dequantize, and a GEMV that copies the GPU's summation order |
| Fake quant | `python/needle/fp4/quantize.py`, backend `floor`/`abs` | quantize-dequantize on any needle device |
| Autograd | `python/needle/ops/ops_fp4.py`, `python/needle/fp4/config.py` | `FP4Matmul` Op; `QuantConfig` says which of the three matmuls are 4-bit, whether SR is on, and how W is quantized (`weights`: per matmul, 16x16 tiles, or SR re-quantization) |
| Layer | `python/needle/nn/nn_fp4.py` | `nn.QuantizedLinear`, a drop-in for `nn.Linear` |
| Real storage | `python/needle/fp4/packed.py`, `src/nvfp4.h`, both backends | `NVFP4Matrix`: packed codes, E4M3 scales, tensor scale; quantize+pack and GEMV kernels |
| Stretch | `python/needle/fp4/formats.py` | NVFP4 vs MXFP4 weight quantization |
| Experiments | `apps/`, `benchmarks/` | training grid, gradient error, packed inference, GEMV benchmark |

**Three matmuls, three axes.** needle stores W as (in, out). Fprop quantizes X along axis 1 and W along axis 0;
Dgrad quantizes dY along axis 1 and W along axis 1; Wgrad quantizes X along axis 0 and dY along axis 0. The backward
pass is a straight-through estimator: the FP4 matmul's gradient is computed by the other two (FP4) matmuls. SR applies
only to the gradient operand dY.

**Fake vs real 4-bit.** Training uses fake quantization: values are rounded to NVFP4, then an FP32 matmul runs. It
reproduces FP4's effect on accuracy exactly but saves nothing. The tests check that it gives the reference's bits on
every device, which is why the multi-seed training grids could run on CPU. Inference uses real storage: two codes per byte (element 2i in the low nibble, sign in
bit 3), one E4M3 byte per 16 values.

**Bit-exact everywhere.** Float32 operations run in the reference's order; the GPU uses `_rn` intrinsics and the CPU
is built with `-ffp-contract=off` so no fused multiply-add changes a result. **Kernels:** quantize+pack uses one
thread per element, a half-warp `__shfl_xor_sync` reduction for each block's amax, and `__shfl_down_sync` to pair
codes into bytes. The GEMV gives each warp four output rows: lanes read 8-byte chunks of codes, decode through
shared-memory tables, load each block of x once with float4 loads and reuse it for all four rows, and combine lane
sums with a butterfly whose order the NumPy reference reproduces.

In [ ]:
# Which branch of the repository to run: "main", or a dev/<name>-<topic> branch to try unmerged work.
BRANCH = "main"

In [ ]:
import base64, os, subprocess, sys

# Where to work: Google Drive on Colab (results survive the session), the current folder anywhere else.
try:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    WORK = '/content/drive/MyDrive/10714'
except ImportError:
    userdata, WORK = None, os.getcwd()

# Private repo: create a GitHub token with read access, then in Colab click the key icon (Secrets), add a
# secret named GITHUB_TOKEN and allow notebook access (elsewhere: set the GITHUB_TOKEN environment variable).
# The token is passed to git for each command and never saved.
try:
    token = userdata.get('GITHUB_TOKEN') if userdata else os.environ.get('GITHUB_TOKEN')
except Exception:
    token = None
basic = base64.b64encode(f"x-access-token:{token}".encode()).decode() if token else ""
AUTH = f"-c http.extraheader='Authorization: Basic {basic}'" if token else ""

if not os.path.exists('python/needle/fp4'):        # not already inside the repository
    os.makedirs(WORK, exist_ok=True)
    os.chdir(WORK)
    if not os.path.exists('fp4-needle'):
        !git {AUTH} clone -q https://github.com/shreya-v7/fp4-needle.git
    os.chdir('fp4-needle')
# Match GitHub exactly (Colab autosaves this notebook into the Drive copy, which would block an update).
# Only tracked files are reset; results/, data/ and build/ are git-ignored and kept.
!git {AUTH} fetch -q origin && git checkout -q -f {BRANCH} && git reset -q --hard origin/{BRANCH}
head = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
remote = subprocess.check_output(['git', 'rev-parse', f'origin/{BRANCH}'], text=True).strip()
assert head == remote, f"Checkout is at {head[:7]} but origin/{BRANCH} is {remote[:7]}; see the git output above."
print(f"Running branch {BRANCH} at commit {head[:7]} in {os.getcwd()}")

In [ ]:
# Build needle's C++ (CPU) and, when a GPU and CUDA are present, CUDA backends.
!pip3 install -q pybind11
!make clean > /dev/null && make 2>&1 | tail -2
os.environ['PYTHONPATH'] = './python'    # for the scripts run below with !python3
os.environ['NEEDLE_BACKEND'] = 'nd'
sys.path += ['python', 'apps']
import csv
from IPython.display import Image, Markdown, display
import needle as ndl

DEVICE = 'cuda' if ndl.cuda().enabled() else 'cpu'
GPU = DEVICE == 'cuda'
R = 'results/colab'                      # everything this run produces goes here
os.makedirs(R, exist_ok=True)
if GPU:
    !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
    print('Running on the GPU.')
else:
    print('No GPU: running on the CPU. The CUDA tests and the GPU benchmark are skipped; everything else runs.')

## 5. Evaluation

### 5.1 Correctness

`tests/fp4` checks every piece against the reference: E2M1 and E4M3 rounding at every grid point and tie, packing
layout, padding, SR being unbiased, the straight-through gradient, all three matmul placements, the quantize+pack,
dequantize and GEMV kernels on CPU and CUDA, and the MXFP4 quantizer. CI runs the CPU half on every pull request.

In [ ]:
!python3 -m pytest tests/fp4 -q 2>&1 | tee {R}/tests.txt | tail -2
last = open(f'{R}/tests.txt').read().strip().splitlines()[-1]
ok = 'failed' not in last and 'error' not in last
print('\nVerdict:', 'all tests pass.' if ok else 'SOME TESTS FAILED (see above).',
      '' if GPU else 'The skipped tests are the CUDA ones; they run when a GPU is present.')

### 5.2 Which matmul breaks training?

A 3-layer MLP (input, 256, 256, 10), every linear layer a `QuantizedLinear`, SGD (lr 0.05, momentum 0.9), batch 128,
3 epochs. Every configuration starts from the same weights and sees the same batches, so runs differ only by FP4.

**CIFAR-10** (all 50,000 training images, 4 seeds, mean ± std):

| FP4 matmuls | test accuracy (%) | train loss (last 100 steps) |
|---|---|---|
| FP32 | 43.96 ± 0.45 | 1.584 |
| Fprop | 43.71 ± 0.84 | 1.590 |
| **Dgrad** | 36.41 ± 3.22 | 1.716 |
| Dgrad + SR | 43.53 ± 0.56 | 1.592 |
| Wgrad | 44.40 ± 0.81 | 1.587 |
| Wgrad + SR | 44.11 ± 0.14 | 1.586 |
| Fprop + Dgrad | 38.50 ± 1.82 | 1.737 |
| Fprop + Dgrad + SR | 43.94 ± 0.63 | 1.604 |
| **All three** | 23.34 ± 3.99 | 2.766 |
| All three + SR | 43.77 ± 0.31 | 1.612 |

1. **Fprop in 4 bits is harmless**: within seed noise of FP32.
2. **Dgrad with round-to-nearest costs 7.5 points**, on every seed. Its error flows back into
   every earlier layer's gradient.
3. **Wgrad alone is harmless**, but on top of Dgrad it pushes training loss up instead of down.
4. **Stochastic rounding removes every gap.**

An earlier run on 20,000 images with 3 seeds gave the same picture (`docs/results/cpu_cifar/`). **MNIST** is too easy
to separate the configurations: all are within 0.4 points of FP32's 94.90%, though all three with round-to-nearest
again has the highest training loss (0.197 vs 0.183) and SR brings it back (0.181) (`docs/results/cpu_sgd/`).

In [ ]:
display(Image('docs/results/cpu_cifar_full/cifar_ablation.png'))
display(Image('docs/results/cpu_cifar_full/cifar_grid.png'))
display(Image('docs/results/cpu_cifar_weights/cifar_weights.png'))

### 5.3 What 4-bit rounding does to the gradients

`apps/fp4_grad_error.py` freezes an FP32 model and compares each configuration's weight gradients with the exact
ones on a fixed batch (first two layers, CIFAR-10, after one epoch of training). Round-to-nearest gives the same
wrong gradient every time, so all its error is **bias**. For SR it averages 32 draws and splits the error into bias
(what averaging cannot remove) and **noise** (what SGD averages away over many steps).

| configuration | error (%) | bias (%) | noise (%) | cosine with exact gradient |
|---|---|---|---|---|
| Fprop, round to nearest | 35.3 | 35.3 | 0.0 | 0.952 |
| Dgrad, round to nearest | 15.9 | 15.9 | 0.0 | 0.989 |
| Dgrad + SR | 14.0 | 11.2 | 10.4 | 0.990 |
| Dgrad, only dY rounded, to nearest | 11.3 | 11.3 | 0.0 | 0.995 |
| Dgrad, only dY rounded, SR | 8.9 | 2.4 | 10.1 | 0.996 |
| Dgrad, only W rounded | 11.0 | 11.0 | 0.0 | 0.994 |

Two things follow. **SR on dY works as theory says:** it turns most of dY's bias into noise. **Error size alone does
not predict harm:** rounding W leaves as much bias as rounding dY, and Fprop's error is the largest of all, yet
section 5.2 shows Fprop is harmless. Which rounding actually hurts training has to be measured by training, which
the next two sections do. The other checkpoints (before training, after 3 epochs) show the same pattern
(`docs/results/grad_error_cifar/`).

### 5.4 Is it the gradient rounding or the weights?

There is a second suspect. Dgrad re-quantizes W along a different axis than Fprop does, so the backward pass
multiplies by a slightly different weight than the forward pass used. NVIDIA reports that "errors induced from
inconsistent weights materialize in the activation gradients" and fixes it with one scale per 16x16 tile of W [1];
Thinking Machines instead re-quantizes the forward weight with SR, so the two agree in expectation [8]. We added both
as a `QuantConfig.weights` option and trained all-FP4 models on CIFAR-10 (4 seeds):

| gradient dY rounding | weight quantization for Fprop and Dgrad | from | test accuracy (%) | train loss |
|---|---|---|---|---|
| FP32 |  |  | 43.96 ± 0.45 | 1.584 |
| round to nearest | 1x16, separately per matmul | this project | 23.34 ± 3.99 | 2.766 |
| round to nearest | 16x16 tiles, shared | NVIDIA [1] | 19.80 ± 2.99 | 2.792 |
| round to nearest | forward W re-quantized with SR | Thinking Machines [8] | 20.82 ± 5.34 | 2.477 |
| stochastic | 1x16, separately per matmul | this project | 43.77 ± 0.31 | 1.612 |
| stochastic | 16x16 tiles, shared | NVIDIA [1] | 43.11 ± 0.45 | 1.609 |
| stochastic | forward W re-quantized with SR | Thinking Machines [8] | 42.99 ± 0.72 | 1.617 |

**It is the gradient rounding.** Making the weight consistent does not rescue round-to-nearest training
(19.8% with tiles, 20.8% with SR re-quantization, against
23.3%), while SR on dY rescues all three weight schemes. At our scale the consistent schemes
even cost a little (43.1% and 43.0% against
43.8%): one scale per 256 values is coarser, and re-quantizing with SR adds noise. Both
papers train models with billions of parameters for trillions of tokens, where small weight inconsistencies have far
longer to add up; on a small MLP for 3 epochs, the bias in dY dominates.

### 5.5 Re-running the key result

The five configurations that carry the result, 1 seed: all 50,000 CIFAR-10 images on a GPU, the first 20,000 on a CPU
so the run stays short. It also saves the FP32 model for section 5.7.

In [ ]:
import tarfile, urllib.request
if not os.path.exists('data/cifar-10-batches-py'):
    os.makedirs('data', exist_ok=True)
    urllib.request.urlretrieve('https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz', 'data/cifar-10-python.tar.gz')
    tarfile.open('data/cifar-10-python.tar.gz').extractall('data')

SUBSET = '' if GPU else '--train-subset 20000'   # all 50,000 images on a GPU, the first 20,000 on a CPU
KEY = 'fp32 dgrad dgrad+sr fprop+dgrad+wgrad fprop+dgrad+wgrad+sr'
!python3 apps/fp4_experiment.py --dataset cifar10 --configs all --only {KEY} --device {DEVICE} --epochs 3 --seed 0 {SUBSET} --out {R}/cifar_check.csv --save-weights "{R}/ckpt/cifar_{{config}}_s{{seed}}.npz"

acc = {r['config']: 100 * float(r['accuracy']) for r in csv.DictReader(open(f'{R}/cifar_check.csv')) if r['split'] == 'test'}
print(f"\nFP32: {acc['fp32']:.1f}%")
for c, name in [('dgrad', 'Dgrad only'), ('fprop+dgrad+wgrad', 'All three')]:
    print(f"{name:11s} round to nearest {acc[c]:5.1f}% ({acc[c] - acc['fp32']:+.1f})   "
          f"with SR {acc[c + '+sr']:5.1f}% ({acc[c + '+sr'] - acc['fp32']:+.1f})")
holds = all(acc[c] < acc[c + '+sr'] - 1 for c in ('dgrad', 'fprop+dgrad+wgrad'))
print('Verdict:', 'the hypothesis holds on this run: round-to-nearest gradients cost accuracy and SR wins it back.'
      if holds else 'this single run does not separate the configurations clearly; the 4-seed results above are the evidence.')

### 5.6 Real 4-bit weights: memory and GEMV speed

At batch size 1 every layer is a matrix-vector product (GEMV), limited by reading the weights. The benchmark
compares our FP32 GEMV with our NVFP4 GEMV, which reads the packed bytes directly, and the homework matmul. The second
command trains an FP32 MNIST MLP, packs its weights and runs it from the packed form.

In [ ]:
import hashlib, urllib.request

# MNIST files and the MD5 of the copy our homeworks used (identical to the course repo).
MNIST = {"train-images-idx3-ubyte.gz": "0cb5796cd8e7c7bffef03edbdb01fac5",
         "train-labels-idx1-ubyte.gz": "23e0cdb926d142c2d470a3641a19d1ff",
         "t10k-images-idx3-ubyte.gz":  "3d4db374c734e4f92d346e063bf6e41a",
         "t10k-labels-idx1-ubyte.gz":  "0407fd84edb9a00bbad73c377ff95de9"}
os.makedirs("data", exist_ok=True)
for name, md5 in MNIST.items():
    dst = os.path.join("data", name)
    if not os.path.exists(dst):
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/dlsyscourse/hw0/main/data/{name}", dst)
    assert hashlib.md5(open(dst, "rb").read()).hexdigest() == md5, f"{dst} is corrupted; delete it and rerun"
print("MNIST ok")

In [ ]:
# Square matrices, plus (on a GPU) the rectangular MLP weights of a 7B-class LLM: up 11008x4096, down 4096x11008.
SHAPES = '--sizes 1024 2048 4096 8192 --shapes 11008x4096 4096x11008' if GPU else '--sizes 1024 2048 4096'
!python3 benchmarks/bench_gemv.py --device {DEVICE} {SHAPES} --out {R}/bench_gemv.csv
display(Image(f'{R}/bench_gemv.png'))
rows = {r['kernel']: r for r in csv.DictReader(open(f'{R}/bench_gemv.csv')) if r['shape'] == '4096x4096'}
fp32, fp4 = rows['fp32 gemv (ours)'], rows['nvfp4 gemv (ours)']
print(f"\nVerdict at 4096 x 4096: weights {float(fp32['weight_MiB']):.0f} MiB -> {float(fp4['weight_MiB']):.1f} MiB; "
      f"NVFP4 GEMV runs at {float(fp32['time_us']) / float(fp4['time_us']):.2f}x the speed of our FP32 GEMV"
      + (f" ({float(fp4['pct_of_peak']):.0f}% vs {float(fp32['pct_of_peak']):.0f}% of peak bandwidth)." if GPU
         else " (CPU wall-clock; the GPU numbers are the ones that matter)."))

# MNIST MLP run from packed 4-bit weights: memory per layer, test accuracy, batch-1 latency
!python3 apps/fp4_inference.py --device {DEVICE} 2>&1 | tee {R}/inference.txt

Packed weights are 7.0x smaller (9 MiB instead of 64 MiB at 4096 x 4096), and the packed MNIST MLP scored 97.42%
against 97.54% in FP32 in our CPU run (`docs/results/cpu_inference/`). In our first GPU run (a T4), our FP32 GEMV reached 84% of peak bandwidth, 4x
faster than the homework matmul, but our first NVFP4 kernel was slower than it (0.59x at 4096 x 4096) despite reading
7x fewer bytes: it loaded x one checked float at a time, decoded scales with `ldexpf`, and ran one long chain of
dependent additions per thread. The version measured above loads x once per four rows, decodes through shared-memory
tables, and keeps four rows' sums in flight, with the order of additions unchanged. On a GPU, the verdict line above
gives its speed; on a CPU the timings only show that the code runs.

### 5.7 Stretch goal: MXFP4 vs NVFP4

We quantize every weight of a trained FP32 MLP in each format, then measure the error (SQNR = -10 log10(squared
error / weight power), higher is better) and the test accuracy with the quantized weights (`docs/results/mxfp4/`):

| FP32 model | NVFP4, 4.5 bits: SQNR / accuracy | MXFP4, 4.25 bits: SQNR / accuracy | FP32 accuracy |
|---|---|---|---|
| MNIST MLP | 19.81 dB / 95.08% | 18.88 dB / 94.48% | 94.93% |
| CIFAR-10 MLP | 19.85 dB / 43.90% | 19.04 dB / 43.67% | 44.04% |

NVFP4's 16-value blocks and E4M3 scales give about 0.8 to 0.9 dB less error than MXFP4's 32-value power-of-two
blocks, for 0.25 more bits per weight; on these small models both keep accuracy within 0.5 points. The cell below
repeats the comparison on the CIFAR-10 model trained in section 5.5.

In [ ]:
!python3 apps/fp4_mxfp4.py {R}/ckpt/cifar_fp32_s0.npz --device {DEVICE} --out {R}/mxfp4.md

## 6. Discussion and limitations

The hypothesis holds, with a sharper answer than we expected: the matmul that breaks is Dgrad under round-to-nearest.
Its biased error is passed back through every earlier layer, while SR turns it into noise that SGD averages away. Fprop
tolerates 4 bits, and Wgrad does on its own. A practical recipe follows: round-to-nearest forward, SR for gradients.

* **Scale.** needle runs at about 30 GFLOP/s, so we train small MLPs. Effects that appear at billions of parameters
  and trillions of tokens are out of reach.
* **Fake quantization.** Training measures accuracy, not speed; only inference uses real 4-bit storage.
* **Short training.** 3 epochs; longer training could change the size of the gaps, though the Dgrad effect appears in
  every seed of both CIFAR-10 runs.
* **SR is not free of bias.** W and X are still rounded to nearest in the backward matmuls (section 5.3).

## 7. References

1. NVIDIA. [Pretraining Large Language Models with NVFP4](https://arxiv.org/abs/2509.25149). 2025.
2. Chmiel, Fishman, Banner, Soudry. [FP4 All the Way: Fully Quantized Training of LLMs](https://arxiv.org/abs/2505.19115). 2025.
3. Tseng, Yu, Park. [Training LLMs with MXFP4](https://arxiv.org/abs/2502.20586). 2025.
4. Castro et al. [Quartet: Native FP4 Training Can Be Optimal for Large Language Models](https://arxiv.org/abs/2505.14669). 2025.
5. DeepSeek-AI. [DeepSeek-V3 Technical Report](https://arxiv.org/abs/2412.19437). 2024.
6. Open Compute Project. OCP Microscaling Formats (MX) Specification v1.0. 2023.
7. Datasets: MNIST (LeCun et al., 1998); CIFAR-10 (Krizhevsky, 2009).
8. Cook and Thinking Machines Lab. [End-to-End 4-Bit Training](https://thinkingmachines.ai/blog/end-to-end-4-bit-training/). 2026.

## 8. AI usage statement

| Item | Detail |
|---|---|
| Tool | Claude Code (Anthropic), cloud sessions |
| Model and version | TODO (team): the model name and version shown in our Claude Code session |
| Used for | writing code, tests and experiment scripts; debugging (CI, Colab, data paths); finding related work; drafting this report and the README |
| Not used for | the homework needle code we started from, choosing the project, final judgment on claims |

**Affected files.** Every AI-assisted file or edited section starts with the line "AI-assisted: written with Claude
(Anthropic) via Claude Code; reviewed and tested by the team.": all of `python/needle/fp4/`, `ops/ops_fp4.py`,
`nn/nn_fp4.py`; the NVFP4, `floor` and `abs` sections of both backends and `ndarray.py`; `src/nvfp4.h`; the FP4 apps
in `apps/`; `benchmarks/bench_gemv.py`; `tests/fp4/`; this notebook; `.github/workflows/ci.yml`; the
`-ffp-contract=off` line in `CMakeLists.txt`.

**How we verified it.** (1) A NumPy reference written from the NVFP4 spec; every kernel and op is tested bit for bit
against it. (2) CI runs the tests on every pull request, and nothing reaches main without one. (3) Every number in
this report comes from a run we can repeat with the commands below. (4) Claims were checked against the cited papers.
(5) Each of us read and can explain the code in the parts we present.

## 9. Code and reproduction

Code: our private GitHub repository `shreya-v7/fp4-needle` (link shared with the course staff). Run all reproduces
sections 5.1, 5.5, 5.6 and 5.7. The multi-seed CPU results in 5.2 to 5.4 and 5.7 come from:

```bash
make lib && export PYTHONPATH=python
for s in 0 1 2 3; do
  python3 apps/fp4_experiment.py --dataset cifar10 --configs all --epochs 3 --seed $s --out results/cifar_full_seed$s.csv
  python3 apps/fp4_experiment.py --dataset cifar10 --configs weights --epochs 3 --seed $s --out results/cifar_weights_seed$s.csv
done
python3 apps/plot_fp4.py results/cifar_full_seed*.csv --out-dir results/cifar_full --title "CIFAR-10 MLP" --prefix cifar
python3 apps/fp4_grad_error.py --dataset cifar10 --checkpoints 0 391 1173 --out results/grad_error_cifar.csv
for s in 0 1 2; do python3 apps/fp4_experiment.py --configs all --epochs 3 --seed $s --out results/mnist_seed$s.csv; done
python3 apps/fp4_experiment.py --configs grid --only fp32 --epochs 3 --out results/mnist_fp32.csv --save-weights "results/ckpt/mlp_{config}_s{seed}.npz"
python3 apps/fp4_mxfp4.py results/ckpt/mlp_fp32_s0.npz --out results/mxfp4_mnist.md
```

The last cell zips this run's output; download `fp4_results.zip` from the Files panel (on Colab it is also in
`MyDrive/10714/fp4-needle/` on Drive).

In [ ]:
import zipfile
with zipfile.ZipFile('fp4_results.zip', 'w', zipfile.ZIP_DEFLATED) as z:   # model checkpoints left out
    for folder, _, files in os.walk(R):
        for f in files:
            if not f.endswith('.npz'):
                z.write(os.path.join(folder, f))
print('Created', os.path.abspath('fp4_results.zip'), os.path.getsize('fp4_results.zip') // 1024, 'KB: download it and send it.')